# Core Regressions: Anti-Labor Intensity & Coverage Volume

Tests whether newspaper owners with railroad financial ties produced more anti-labor coverage.
Runs the main regression specifications and coverage volume regressions.

In [61]:
import pandas as pd
import sqlite3
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

DB_PATH = '../data/newspapers.db'
OE_PATH = '../data/personnel_coding/owners_and_editors.csv'
CODE_PATH = '../data/personnel_coding/combined_coding.csv'

# Load intermediate data from 01_data_preparation
df = pd.read_csv('intermediate/analysis_sample.csv')
counts = pd.read_csv('intermediate/sentiment_counts.csv')
paper_year_rr = pd.read_csv('intermediate/paper_year_rr.csv')
person_rr = pd.read_csv('intermediate/person_rr.csv')


oe = pd.read_csv(OE_PATH)

print(f'Loaded analysis sample: {len(df)} newspaper-years')

Loaded analysis sample: 265 newspaper-years


## Regression

In [62]:
# Prepare data
df['year_str'] = df['year'].astype(str)

# Build circulation lookup from master.csv (nearest Ayer directory year)
import numpy as np
master = pd.read_csv('../data/master.csv', low_memory=False)
circ_cols = [c for c in master.columns if 'circulation' in c.lower()]
circ_long = []
for col in circ_cols:
    yr = int(col.split()[0])
    tmp = master[['issn', col]].dropna(subset=['issn', col]).copy()
    tmp = tmp.rename(columns={col: 'circulation'})
    tmp['circ_year'] = yr
    circ_long.append(tmp)
circ_long = pd.concat(circ_long, ignore_index=True)
circ_long['circulation'] = pd.to_numeric(circ_long['circulation'], errors='coerce')
circ_long = circ_long.dropna(subset=['circulation'])
circ_long = circ_long[circ_long['circulation'] > 0]

circ_lookup = {}
for _, row in circ_long.iterrows():
    circ_lookup.setdefault(row['issn'], []).append((row['circ_year'], row['circulation']))

def nearest_circ(issn, year):
    entries = circ_lookup.get(issn)
    if not entries:
        return None
    return min(entries, key=lambda x: abs(x[0] - year))[1]

df['circulation'] = df.apply(lambda r: nearest_circ(r['issn'], r['year']), axis=1)
df['log_circulation'] = np.log(pd.to_numeric(df['circulation'], errors='coerce'))

print(f"Circulation available for {df['log_circulation'].notna().sum()} / {len(df)} obs")

# Sanity check: 10 newspapers with county population
town_lookup = master.set_index('issn')['town'].to_dict()
check = (df.drop_duplicates('issn')[['issn', 'state', 'county_pop', 'log_county_pop']]
         .assign(town=lambda x: x['issn'].map(town_lookup))
         .dropna(subset=['log_county_pop'])
         .head(10)[['town', 'state', 'county_pop', 'log_county_pop']])
print("\nCounty population sanity check:")
print(check.to_string(index=False))

Circulation available for 161 / 265 obs

County population sanity check:
          town        state  county_pop  log_county_pop
       Belfast        MAINE     34316.1       10.443370
      New York     NEW YORK    942292.0       13.756070
    Saint Paul          NaN     41329.0       10.629320
    FORT WORTH        TEXAS     41142.0       10.624785
    Alexandria     VIRGINIA     16755.0        9.726452
    Sacramento   CALIFORNIA     40339.0       10.605074
Salt Lake City          NaN     45217.0       10.719228
     Lancaster PENNSYLVANIA    139447.0       11.845440
        Canton         OHIO     53660.3       10.890429
       Wichita       KANSAS      6392.4        8.762865


### Cross-Sectional Regression (newspaper-level)

Collapses all years into a single observation per newspaper: total anti-labor / total labor articles across the owner's full tenure. N = number of newspapers, not newspaper-years.

In [63]:
# Collapse to one observation per newspaper (sum articles across all years)
df_xs = (
    df.groupby('issn')
    .agg(anti_labor=('anti_labor', 'sum'),
         total_labor=('total_labor', 'sum'),
         railroad_interest=('railroad_interest', 'max'),
         person_republican=('person_republican', 'max'),
         log_circulation=('log_circulation', 'mean'),
         log_county_pop=('log_county_pop', 'mean'))
    .reset_index()
)
df_xs['anti_labor_intensity'] = df_xs['anti_labor'] / df_xs['total_labor']

# OLS (HC3)
m0a = smf.ols('anti_labor_intensity ~ railroad_interest', data=df_xs).fit(cov_type='HC3')

df_xs_pol = df_xs.dropna(subset=['person_republican'])
m0b = smf.ols('anti_labor_intensity ~ railroad_interest + person_republican',
              data=df_xs_pol).fit(cov_type='HC3')

df_xs_pop = df_xs.dropna(subset=['person_republican', 'log_county_pop'])
m0d = smf.ols('anti_labor_intensity ~ railroad_interest + person_republican + log_county_pop',
              data=df_xs_pop).fit(cov_type='HC3')

from statsmodels.iolib.summary2 import summary_col
print("Cross-sectional regressions — OLS (HC3 SEs)")
print()
print(summary_col(
    [m0a, m0b, m0d],
    model_names=['(0a) Bivariate', '(0b) + Party', '(0c) + Party + Pop'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'R²': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'person_republican', 'log_county_pop', 'Intercept']
))

# WLS (weighted by total_labor, HC3)
print("\n\nCross-sectional regressions — WLS weighted by total_labor (HC3 SEs)")
print()
wxa = smf.wls('anti_labor_intensity ~ railroad_interest',
              weights=df_xs['total_labor'], data=df_xs).fit(cov_type='HC3')
wxb = smf.wls('anti_labor_intensity ~ railroad_interest + person_republican',
              weights=df_xs_pol['total_labor'], data=df_xs_pol).fit(cov_type='HC3')
wxd = smf.wls('anti_labor_intensity ~ railroad_interest + person_republican + log_county_pop',
              weights=df_xs_pop['total_labor'], data=df_xs_pop).fit(cov_type='HC3')

print(summary_col(
    [wxa, wxb, wxd],
    model_names=['(0a) Bivariate', '(0b) + Party', '(0c) + Party + Pop'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'R²': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'person_republican', 'log_county_pop', 'Intercept']
))

Cross-sectional regressions — OLS (HC3 SEs)


                  (0a) Bivariate (0b) + Party (0c) + Party + Pop
----------------------------------------------------------------
railroad_interest 0.0802**       0.0955***    0.0972**          
                  (0.0331)       (0.0363)     (0.0384)          
person_republican                0.0623*      0.0731*           
                                 (0.0358)     (0.0391)          
log_county_pop                                -0.0264           
                                              (0.0192)          
Intercept         0.3073***      0.2541***    0.5296***         
                  (0.0233)       (0.0323)     (0.2053)          
R-squared         0.1217         0.2158       0.2567            
R-squared Adj.    0.1017         0.1710       0.1870            
N                 46.0000        38.0000      36.0000           
R²                0.1220         0.2160       0.2570            
Standard errors in parentheses.
* p<.1, ** p

### Panel Regressions (newspaper-year level, OLS, HC3 SEs)

One observation per newspaper-year. Progressively adds year fixed effects, person-level political affiliation, and county population controls.

In [64]:
# Panel OLS regressions (HC3 SEs)
m1 = smf.ols('anti_labor_intensity ~ railroad_interest', data=df).fit(cov_type='HC3')
m2 = smf.ols('anti_labor_intensity ~ railroad_interest + C(year)', data=df).fit(cov_type='HC3')

df_pol = df.dropna(subset=['person_republican']).copy()
m3 = smf.ols('anti_labor_intensity ~ railroad_interest + C(year) + person_republican',
             data=df_pol).fit(cov_type='HC3')

df_pol_pop = df.dropna(subset=['person_republican', 'log_county_pop']).copy()
m4 = smf.ols('anti_labor_intensity ~ railroad_interest + C(year) + person_republican + log_county_pop',
             data=df_pol_pop).fit(cov_type='HC3')

from statsmodels.iolib.summary2 import summary_col
print(f"N: full={len(df)} | +party={len(df_pol)} | +party+pop={len(df_pol_pop)}")
print()
print(summary_col(
    [m1, m2, m3, m4],
    model_names=['(1) Bivariate', '(2) Year FE', '(3) + Party', '(4) + Party + Pop'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'R²': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'person_republican', 'log_county_pop', 'Intercept']
))

N: full=265 | +party=228 | +party+pop=224


                  (1) Bivariate (2) Year FE (3) + Party (4) + Party + Pop
-------------------------------------------------------------------------
railroad_interest 0.0692***     0.0610**    0.0624**    0.0631**         
                  (0.0262)      (0.0258)    (0.0290)    (0.0301)         
person_republican                           0.0980***   0.1123***        
                                            (0.0268)    (0.0309)         
log_county_pop                                          -0.0174          
                                                        (0.0132)         
Intercept         0.2811***     0.1967***   0.1688***   0.3485**         
                  (0.0173)      (0.0620)    (0.0604)    (0.1485)         
C(year)[T.1871]                 -0.0208     -0.0526     -0.0453          
                                (0.0731)    (0.0695)    (0.0694)         
C(year)[T.1872]                 0.1303      0.1069      0.1046      

### Model 4: Editor vs Publisher Split

Runs the main specification (railroad_interest + year FE) separately for newspaper-years where the coded person is an editor vs a publisher.

In [65]:
# Split by role: editors only vs publishers only
df_editors = df[df['has_editor'] == 1].copy()
df_publishers = df[df['has_publisher'] == 1].copy()

print(f"Newspaper-years with editor:    {len(df_editors)}")
print(f"Newspaper-years with publisher: {len(df_publishers)}")

m4_ed = smf.ols('anti_labor_intensity ~ railroad_interest + C(year)', data=df_editors).fit(cov_type='HC3')
m4_pub = smf.ols('anti_labor_intensity ~ railroad_interest + C(year)', data=df_publishers).fit(cov_type='HC3')

print()
print(summary_col(
    [m4_ed, m4_pub],
    model_names=['Editors only', 'Publishers only'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'R²': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'Intercept']
))

Newspaper-years with editor:    227
Newspaper-years with publisher: 203


                  Editors only Publishers only
----------------------------------------------
railroad_interest 0.0434       0.0605*        
                  (0.0290)     (0.0333)       
Intercept         0.1989***    0.1988***      
                  (0.0678)     (0.0738)       
C(year)[T.1871]   -0.0202      -0.0409        
                  (0.0810)     (0.0877)       
C(year)[T.1872]   0.1916*      0.1583         
                  (0.1090)     (0.1127)       
C(year)[T.1873]   0.0454       0.0236         
                  (0.0846)     (0.0865)       
C(year)[T.1876]   0.1203       0.1215         
                  (0.1173)     (0.1284)       
C(year)[T.1877]   0.2738***    0.2801***      
                  (0.0762)     (0.0864)       
C(year)[T.1878]   0.0230       0.0256         
                  (0.0731)     (0.0791)       
C(year)[T.1879]   0.0754       0.0941         
                  (0.0982)     (0

### Robustness: WLS + Clustered Standard Errors

Re-runs the main specifications using WLS (weighted by `total_labor`) and clustering standard errors at the newspaper level to account for within-newspaper serial correlation.

In [66]:
# WLS weighted by total_labor, SEs clustered by newspaper
cluster_kw = {'groups': df['issn']}
cluster_kw_pol = {'groups': df_pol['issn']}
cluster_kw_pop = {'groups': df_pol_pop['issn']}

w1 = smf.wls('anti_labor_intensity ~ railroad_interest',
             weights=df['total_labor'], data=df).fit(cov_type='cluster', cov_kwds=cluster_kw)
w2 = smf.wls('anti_labor_intensity ~ railroad_interest + C(year)',
             weights=df['total_labor'], data=df).fit(cov_type='cluster', cov_kwds=cluster_kw)
w3 = smf.wls('anti_labor_intensity ~ railroad_interest + C(year) + person_republican',
             weights=df_pol['total_labor'], data=df_pol).fit(cov_type='cluster', cov_kwds=cluster_kw_pol)
w4 = smf.wls('anti_labor_intensity ~ railroad_interest + C(year) + person_republican + log_county_pop',
             weights=df_pol_pop['total_labor'], data=df_pol_pop).fit(cov_type='cluster', cov_kwds=cluster_kw_pop)

print("WLS (weighted by total_labor) + newspaper-clustered SEs")
print()
print(summary_col(
    [w1, w2, w3, w4],
    model_names=['(1) Bivariate', '(2) Year FE', '(3) Year+Party', '(4) +County Pop'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'Clusters': lambda m: len(m.cov_kwds['groups'].unique()),
               'R²': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'person_republican', 'log_county_pop', 'Intercept']
))

WLS (weighted by total_labor) + newspaper-clustered SEs


                  (1) Bivariate (2) Year FE (3) Year+Party (4) +County Pop
--------------------------------------------------------------------------
railroad_interest 0.0668***     0.0530***   0.0658***      0.0603**       
                  (0.0209)      (0.0191)    (0.0185)       (0.0245)       
person_republican                           0.0517***      0.0570**       
                                            (0.0180)       (0.0268)       
log_county_pop                                             0.0040         
                                                           (0.0100)       
Intercept         0.2967***     0.2018***   0.1630***      0.1076         
                  (0.0135)      (0.0346)    (0.0338)       (0.1051)       
C(year)[T.1871]                 0.0541      0.0761*        0.0795**       
                                (0.0365)    (0.0396)       (0.0388)       
C(year)[T.1872]                 0.0559    

---

## 6. Coverage Amount Regression

Does railroad financial interest affect the **volume** of labor coverage?

**Outcome:** `labor_coverage_share` = labor articles / total articles per newspaper-year  
**Treatment:** `railroad_interest` (same as above)

In [67]:
# Compute total articles per newspaper-year for ISSNs in the analysis sample
conn = sqlite3.connect(DB_PATH)

analysis_issns = df['issn'].unique().tolist()
placeholders = ','.join(['?'] * len(analysis_issns))

total_articles = pd.read_sql(f"""
    SELECT issn, year, COUNT(*) as total_articles
    FROM articles
    WHERE issn IN ({placeholders})
    GROUP BY issn, year
""", conn, params=analysis_issns)

conn.close()

# Merge: labor article counts (from 'counts' df) + total articles
coverage = counts[['issn', 'year', 'total_labor']].merge(
    total_articles, on=['issn', 'year'], how='inner'
)
coverage['labor_coverage_share'] = coverage['total_labor'] / coverage['total_articles']

# Merge with treatment
df_cov = coverage.merge(paper_year_rr, on=['issn', 'year'], how='inner')
df_cov['year_str'] = df_cov['year'].astype(str)

print(f"Coverage analysis sample: {len(df_cov)} newspaper-years")
print(f"\nLabor coverage share by treatment group:")
print(df_cov.groupby('railroad_interest')['labor_coverage_share'].describe())

Coverage analysis sample: 265 newspaper-years

Labor coverage share by treatment group:
                   count      mean       std       min       25%       50%  \
railroad_interest                                                            
0                  151.0  0.001969  0.001684  0.000147  0.000698  0.001536   
1                  114.0  0.002102  0.001708  0.000120  0.000826  0.001685   

                        75%       max  
railroad_interest                      
0                  0.002668  0.009834  
1                  0.002786  0.009645  


In [68]:
# Merge person_republican and log_county_pop into coverage df
df_cov = df_cov.merge(
    df[['issn', 'year', 'person_republican', 'log_county_pop']],
    on=['issn', 'year'], how='left'
)

# Coverage regressions
c1 = smf.ols('labor_coverage_share ~ railroad_interest', data=df_cov).fit(cov_type='HC3')
c2 = smf.ols('labor_coverage_share ~ railroad_interest + C(year)', data=df_cov).fit(cov_type='HC3')

df_cov_pol = df_cov.dropna(subset=['person_republican'])
c3 = smf.ols('labor_coverage_share ~ railroad_interest + C(year) + person_republican',
             data=df_cov_pol).fit(cov_type='HC3')

df_cov_pop = df_cov.dropna(subset=['person_republican', 'log_county_pop'])
c4 = smf.ols('labor_coverage_share ~ railroad_interest + C(year) + person_republican + log_county_pop',
             data=df_cov_pop).fit(cov_type='HC3')

print(summary_col(
    [c1, c2, c3, c4],
    model_names=['(1) Bivariate', '(2) Year FE', '(3) + Party', '(4) + Party + Pop'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'R²': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'person_republican', 'log_county_pop', 'Intercept']
))


                  (1) Bivariate (2) Year FE (3) + Party (4) + Party + Pop
-------------------------------------------------------------------------
railroad_interest 0.0001        0.0001      0.0001      -0.0001          
                  (0.0002)      (0.0002)    (0.0002)    (0.0002)         
person_republican                           0.0000      -0.0000          
                                            (0.0002)    (0.0002)         
log_county_pop                                          0.0003***        
                                                        (0.0001)         
Intercept         0.0020***     0.0009***   0.0010***   -0.0026***       
                  (0.0001)      (0.0002)    (0.0003)    (0.0009)         
C(year)[T.1871]                 0.0006      0.0007      0.0005           
                                (0.0007)    (0.0009)    (0.0009)         
C(year)[T.1872]                 0.0006      0.0005      0.0005           
                                (0.00